# 2021 CDC Natality Data

Every year, millions of babies are born in the United States. Anonymized data surrounding each birth is collected by the CDC and made available to the public. Here we will explore that data.

### Common Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 1. Load Data and Clean

In [ ]:
df = pd.read_parquet("../../data/processed/natality_2021_numeric.parquet")

In [ ]:
df.head()

In [ ]:
df.info()

We have too many features to easily view null counts. We know some columns from the user guide will not be useful, so we'll start by dropping those.

In [ ]:
flag_column_names = df.filter(regex="(.*imputed.*|.*flag.*)$", axis=1).columns

flag_column_names_list = flag_column_names.to_list()

df = df.drop(flag_column_names_list, axis="columns")

Our parquet file also includes two columns that are just filled by "2021" entries. We'll drop those as well.

In [ ]:
df = df.drop(["birth_year", "2021"], axis="columns")

In [ ]:
df.info()

That cut our features down to a more manageable 144. That's still too many to easily view our null values using `info()`, so we'll use another method to examine our null values.

In [ ]:
null_counts_df = df.isnull().sum().reset_index()
null_counts_df.columns = ["column_name", "null_count"]

null_counts_df

In [ ]:
columns_with_nulls = null_counts_df[null_counts_df["null_count"] > 0][
    "column_name"
].tolist()

print("Number of columns with nulls:", len(columns_with_nulls))

We have 56 columns with null values. Some have significantly more nulls than others, so different methods will need to be applied to remove or impute those values.

In [ ]:
null_counts_df.sort_values("null_count", ascending=False)

In [ ]:
sns.barplot(
    x="null_count",
    y="column_name",
    data=null_counts_df.sort_values("null_count", ascending=False).head(20),
)
plt.title("Top 20 Columns with Most Null Values")
plt.xlabel("Number of Null Values")
plt.ylabel("Column Name")
plt.show()

We'll opt to drop the top five columns as they are more null values than recorded values.

In [ ]:
df = df.drop(
    [
        "reported_mothers_age_used",
        "assistive_reproductive_technology",
        "fertility_enhancing_drugs",
        "trial_of_labor_attempted_if_cesarean",
        "paternity_acknowledged",
    ],
    axis="columns",
)

The remainder of our columns with null values can be addressed by imputation. We'll deal with each column on an as-needed basis in targeted analysis.

Finally, we'll save out our cleaned dataset as a new parquet file.

In [ ]:
df.to_parquet("../../data/processed/natality_2021_cleaned.parquet", index=False)